# LAB | Extractive Question Answering

<div style="border:1px solid #ccc; border-radius:6px; text-align: center;">

<br>

**Run this notebook in [Google Colab](https://colab.research.google.com/) for GPU acceleration.**

<br>

</div>

This notebook demonstrates how Pinecone helps you build an extractive question-answering application. To build an extractive question-answering system, we need three main components:

- A vector index to store and run semantic search
- A retriever model for embedding context passages
- A reader model to extract answers

We will use the SQuAD dataset, which consists of **questions** and **context** paragraphs containing question **answers**. We generate embeddings for the context passages using the retriever, index them in the vector database, and query with semantic search to retrieve the top k most relevant contexts containing potential answers to our question. We then use the reader model to extract the answers from the returned contexts.

Let's get started by installing the packages needed for notebook to run:

# Install Dependencies

In [1]:
!pip install -q \
    transformers==4.46.3 \
    datasets==4.5.0 \
    pinecone==5.4.2 \
    sentence-transformers==3.3.1 \
    huggingface_hub==0.26.5

!rm -rf ~/.cache/huggingface

In [5]:
import torch

print("PyTorch:", torch.__version__)
print("GPU disponibile:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu130
GPU disponibile: True
GPU: Tesla T4


In [3]:
%pip install pinecone==5.4.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [pinecone]


In [2]:
import transformers
import datasets
import pinecone
import sentence_transformers
import huggingface_hub

print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("pinecone:", pinecone.__version__)
print("sentence-transformers:", sentence_transformers.__version__)
print("huggingface-hub:", huggingface_hub.__version__)

transformers: 4.46.3
datasets: 4.5.0
pinecone: 5.4.2
sentence-transformers: 3.3.1
huggingface-hub: 0.26.5


In [1]:
import sys
print(sys.version)

3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]


In [3]:
from datasets import load_dataset

df = load_dataset("rajpurkar/squad", split="train").to_pandas()

print("Dimensioni iniziali:", df.shape)
df.head()

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

Dimensioni iniziali: (87599, 5)


,id,title,context,question,answers
0,5733be284776f41900661182,University_of_Notre_Dame,"Architecturally, the school has a Catholic cha...",To whom did the Virgin Mary allegedly appear i...,"{'text': ['Saint Bernadette Soubirous'], 'answ..."
1,5733be284776f4190066117f,University_of_Notre_Dame,"Architecturally, the school has a Catholic cha...",What is in front of the Notre Dame Main Building?,"{'text': ['a copper statue of Christ'], 'answe..."
2,5733be284776f41900661180,University_of_Notre_Dame,"Architecturally, the school has a Catholic cha...",The Basilica of the Sacred heart at Notre Dame...,"{'text': ['the Main Building'], 'answer_start'..."
3,5733be284776f41900661181,University_of_Notre_Dame,"Architecturally, the school has a Catholic cha...",What is the Grotto at Notre Dame?,{'text': ['a Marian place of prayer and reflec...
4,5733be284776f4190066117e,University_of_Notre_Dame,"Architecturally, the school has a Catholic cha...",What sits on top of the Main Building at Notre...,{'text': ['a golden statue of the Virgin Mary'...


In [ ]:
# Force-kill the kernel process to restart it, ensuring the newly
# installed/uninstalled package versions are loaded in a fresh process

import os
import sys

print(
    "⚠️ WARNING:\n"
    "This code cell will restart the kernel to load the newly installed packages...\n"
    "If you used 'Run all', it will stop here — that's expected.\n"
    "See the note below on how to continue running the rest of the notebook.",
    flush=True,
)
sys.stdout.flush()

os.kill(os.getpid(), 9)

⚠️ WARNING:
This code cell will restart the kernel to load the newly installed packages...
If you used 'Run all', it will stop here — that's expected.
See the note below on how to continue running the rest of the notebook.


<div style="border:1px solid #ccc; border-radius:6px; padding:12px;">

⚠️ **Do not use "Run all" for this whole notebook in one go.** The cell above force-restarts the kernel (`os.kill`) so the newly installed packages load cleanly. Colab's "Run all" cannot continue past a kernel restart — it will always stop right after that cell, no matter how many times you retry it.

Instead, run it in two steps:
1. Run only the two cells above (install + restart). Wait for the kernel to finish restarting (a few seconds).
2. Click on the next cell below, then use **Runtime → Run after** (or manually run the remaining cells one by one / select them and press Shift+Enter). Do not click "Run all" again, since it would just re-trigger the restart at the same spot.

</div>

# Load Dataset

Now let's load the SQUAD dataset from the HuggingFace Model Hub. We load the dataset into a pandas dataframe and filter the title, question, and context columns, and we drop any duplicate context passages.

In [4]:
from datasets import load_dataset

# load the squad dataset into a pandas dataframe
df = load_dataset("rajpurkar/squad", split="train").to_pandas()

In [ ]:
# select only title and context column
df = None
# drop rows containing duplicate context passages
df = None
df

In [5]:
# select only title and context column
df = df[["title", "context"]]

# drop rows containing duplicate context passages
df = df.drop_duplicates(subset=["context"])

df

,title,context
0,University_of_Notre_Dame,"Architecturally, the school has a Catholic cha..."
5,University_of_Notre_Dame,"As at most other universities, Notre Dame's st..."
10,University_of_Notre_Dame,The university is the major seat of the Congre...
15,University_of_Notre_Dame,The College of Engineering was established in ...
20,University_of_Notre_Dame,All of Notre Dame's undergraduate students are...
...,...,...
87574,Kathmandu,"Institute of Medicine, the central college of ..."
87579,Kathmandu,Football and Cricket are the most popular spor...
87584,Kathmandu,The total length of roads in Nepal is recorded...
87589,Kathmandu,The main international airport serving Kathman...


# Pinecone Setup

Pinecone is a managed vector database that lets us store context embeddings and run fast semantic search over them. In this section, we set up the connection to Pinecone using an API key and create the index that will hold our context embeddings.

In [6]:
import os
import getpass
from pinecone import Pinecone

#
# For this notebook, you'll need a Pinecone API key,
# to get one, sign up for free at https://app.pinecone.io/,
# then go to "API Keys" and copy the default key (or create a new one).
#
# Once you have your Pinecone API key,
# load it from Colab's "Secrets" manager
# (the key icon in the left sidebar).
# Add a secret named PINECONE_API_KEY there and enable notebook access.
#
# Note: Colab Secrets rely on a UI popup to grant access, which only works when
# running in the Colab UI itself. If you're running this notebook through
# another editor (e.g. VS Code's Colab extension), that popup isn't available,
# so we fall back to an environment variable or a secure manual prompt instead.
api_key = os.environ.get('PINECONE_API_KEY')


if not api_key:
    try:
        from google.colab import userdata
        api_key = userdata.get('PINECONE_API_KEY')
    except Exception:
        api_key = None

if not api_key:
    api_key = getpass.getpass('Enter your Pinecone API key: ')



<br>

Once we have the API key, we can initialize the connection:

In [7]:
from pinecone import Pinecone, ServerlessSpec

spec = ServerlessSpec(
    cloud="aws", region="us-east-1"
)

# connect to pinecone environment
pc = Pinecone(
    api_key = api_key
)

Now we create a new index called "question-answering" — we can name the index anything we want. We specify the metric type as "cosine" and dimension as 384 because the retriever we use to generate context embeddings is optimized for cosine similarity and outputs 384-dimension vectors.

In [ ]:
index_name = None

# check if the extractive-question-answering index exists
if index_name not in pinecone.list_indexes().names():
    # create the index if it does not exist
    None
# connect to extractive-question-answering index we created
index = pinecone.Index(index_name)

In [8]:
index_name = "question-answering"

# Check if the index exists
if index_name not in pc.list_indexes().names():
    # Create the index if it does not exist
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=spec
    )

# Connect to the index
index = pc.Index(index_name)

# Initialize Retriever

Next, we need to initialize our retriever. The retriever will mainly do two things:

- Generate embeddings for all context passages (context vectors/embeddings)
- Generate embeddings for our questions (query vector/embedding)

The retriever will generate embeddings in a way that the questions and context passages containing answers to our questions are nearby in the vector space. We can use cosine similarity to calculate the similarity between the query and context embeddings to find the context passages that contain potential answers to our question.

We will use a SentenceTransformer model named ``multi-qa-MiniLM-L6-cos-v1`` designed for semantic search and trained on 215M (question, answer) pairs from diverse sources as our retriever.

In [9]:
import torch
from sentence_transformers import SentenceTransformer

# set device to GPU if available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# load the retriever model from huggingface model hub
retriever = SentenceTransformer(
    'multi-qa-MiniLM-L6-cos-v1',
    device=device
) #use the 'multi-qa-MiniLM-L6-cos-v1' model from HuggingFace to build the retriever

retriever

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)

# Generate Embeddings and Upsert

Next, we need to generate embeddings for the context passages. We will do this in batches to help us more quickly generate embeddings and upload them to the Pinecone index. When passing the documents to Pinecone, we need an id (a unique value), context embedding, and metadata for each document representing context passages in the dataset. The metadata is a dictionary containing data relevant to our embeddings, such as the article title, context passage, etc.

In [ ]:
from tqdm.auto import tqdm

# we will use batches of 64
batch_size = 64

for i in tqdm(range(0, len(df), batch_size)):
    # find end of batch
    None
    # extract batch
    None
    # generate embeddings for batch
    emb = None
    # get metadata
    meta = None
    # create unique IDs
    ids = None
    # add all to upsert list
    to_upsert = None
    # upsert/insert these records to pinecone
    _ = index.upsert(vectors=to_upsert)

# check that we have all vectors in index
index.describe_index_stats()

In [10]:

from tqdm.auto import tqdm

# we will use batches of 64
batch_size = 64

for i in tqdm(range(0, len(df), batch_size)):
    # find end of batch
    i_end = min(i + batch_size, len(df))

    # extract batch
    batch = df.iloc[i:i_end]

    # generate embeddings for batch
    emb = retriever.encode(batch["context"].tolist()).tolist()

    # get metadata
    meta = batch.to_dict("records")

    # create unique IDs
    ids = [str(x) for x in range(i, i_end)]

    # add all to upsert list
    to_upsert = list(zip(ids, emb, meta))

    # upsert/insert these records to pinecone
    _ = index.upsert(vectors=to_upsert)

# check that we have all vectors in index
index.describe_index_stats()


  0%|          | 0/296 [00:00<?, ?it/s]

{'dimension': 384,
 'index_fullness': 0.0,
 'namespaces': {'': {'vector_count': 18891}},
 'total_vector_count': 18891}

# Initialize Reader

We use the `deepset/electra-base-squad2` model from the HuggingFace model hub as our reader model. We load this model into a "question-answering" pipeline from HuggingFace transformers and feed it our questions and context passages individually. The model gives a prediction for each context we pass through the pipeline.

In [11]:
from transformers import pipeline

model_name = 'deepset/electra-base-squad2'
# load the reader model into a question-answering pipeline

device_id = 0 if torch.cuda.is_available() else -1
reader = pipeline(tokenizer=model_name, model=model_name, task='question-answering', device=device_id)

reader

config.json:   0%|          | 0.00/635 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Now all the components we need are ready. Let's write some helper functions to execute our queries. The `get_context` function retrieves the context embeddings containing answers to our question from the Pinecone index, and the `extract_answer` function extracts the answers from these context passages.

In [ ]:
# gets context passages from the pinecone index
def get_context(question, top_k):
    # generate embeddings for the question
    xq = None
    # search pinecone index for context passage with the answer
    xc = None
    # extract the context passage from pinecone search result
    c = None
    return c



In [12]:

# gets context passages from the pinecone index
def get_context(question, top_k):
    # generate embeddings for the question
    xq = retriever.encode([question]).tolist()[0]

    # search pinecone index for context passage with the answer
    xc = index.query(
        vector=xq,
        top_k=top_k,
        include_metadata=True
    )

    # extract the context passage from pinecone search result
    c = [match.metadata["context"] for match in xc.matches]

    return c


In [ ]:
from pprint import pprint

# extracts answer from the context passage
def extract_answer(question, context):
    results = []
    for c in context:
        # feed the reader the question and contexts to extract answers
        answer = reader(question=question, context=c)
        # add the context to answer dict for printing both together
        answer["context"] = c
        results.append(answer)
    # sort the result based on the score from reader model
    sorted_result = pprint(sorted(results, key=lambda x: x['score'], reverse=True))
    return sorted_result

In [13]:

from pprint import pprint

# extracts answer from the context passage
def extract_answer(question, context):
    results = []
    for c in context:
        # feed the reader the question and contexts to extract answers
        answer = reader(question=question, context=c)

        # add the context to answer dict for printing both together
        answer["context"] = c
        results.append(answer)

    # sort the result based on the score from reader model
    sorted_result = sorted(
        results,
        key=lambda x: x["score"],
        reverse=True
    )

    pprint(sorted_result)
    return sorted_result


In [14]:
question = "How much oil is Egypt producing in a day?"
context = get_context(question, top_k = 1)
context

['Egypt was producing 691,000 bbl/d of oil and 2,141.05 Tcf of natural gas (in 2013), which makes Egypt as the largest oil producer not member of the Organization of the Petroleum Exporting Countries (OPEC) and the second-largest dry natural gas producer in Africa. In 2013, Egypt was the largest consumer of oil and natural gas in Africa, as more than 20% of total oil consumption and more than 40% of total dry natural gas consumption in Africa. Also, Egypt possesses the largest oil refinery capacity in Africa 726,000 bbl/d (in 2012). Egypt is currently planning to build its first nuclear power plant in El Dabaa city, northern Egypt.']

As we can see, the retiever is working fine and gets us the context passage that contains the answer to our question. Now let's use the reader to extract the exact answer from the context passage.

In [15]:
extract_answer(question, context)

[{'answer': '691,000 bbl/d',
  'context': 'Egypt was producing 691,000 bbl/d of oil and 2,141.05 Tcf of '
             'natural gas (in 2013), which makes Egypt as the largest oil '
             'producer not member of the Organization of the Petroleum '
             'Exporting Countries (OPEC) and the second-largest dry natural '
             'gas producer in Africa. In 2013, Egypt was the largest consumer '
             'of oil and natural gas in Africa, as more than 20% of total oil '
             'consumption and more than 40% of total dry natural gas '
             'consumption in Africa. Also, Egypt possesses the largest oil '
             'refinery capacity in Africa 726,000 bbl/d (in 2012). Egypt is '
             'currently planning to build its first nuclear power plant in El '
             'Dabaa city, northern Egypt.',
  'end': 33,
  'score': 0.9999852180480957,
  'start': 20}]


[{'score': 0.9999852180480957,
  'start': 20,
  'end': 33,
  'answer': '691,000 bbl/d',
  'context': 'Egypt was producing 691,000 bbl/d of oil and 2,141.05 Tcf of natural gas (in 2013), which makes Egypt as the largest oil producer not member of the Organization of the Petroleum Exporting Countries (OPEC) and the second-largest dry natural gas producer in Africa. In 2013, Egypt was the largest consumer of oil and natural gas in Africa, as more than 20% of total oil consumption and more than 40% of total dry natural gas consumption in Africa. Also, Egypt possesses the largest oil refinery capacity in Africa 726,000 bbl/d (in 2012). Egypt is currently planning to build its first nuclear power plant in El Dabaa city, northern Egypt.'}]

The reader model predicted with 99% accuracy the correct answer *691,000 bbl/d* as seen from the context passage. Let's run few more queries.

In [16]:
question = "What are the first names of the men that invented youtube?"
context = get_context(question, top_k=1)
extract_answer(question, context)

[{'answer': 'Hurley and Chen',
  'context': 'According to a story that has often been repeated in the media, '
             'Hurley and Chen developed the idea for YouTube during the early '
             'months of 2005, after they had experienced difficulty sharing '
             "videos that had been shot at a dinner party at Chen's apartment "
             'in San Francisco. Karim did not attend the party and denied that '
             'it had occurred, but Chen commented that the idea that YouTube '
             'was founded after a dinner party "was probably very strengthened '
             'by marketing ideas around creating a story that was very '
             'digestible".',
  'end': 79,
  'score': 0.9999276399612427,
  'start': 64}]


[{'score': 0.9999276399612427,
  'start': 64,
  'end': 79,
  'answer': 'Hurley and Chen',
  'context': 'According to a story that has often been repeated in the media, Hurley and Chen developed the idea for YouTube during the early months of 2005, after they had experienced difficulty sharing videos that had been shot at a dinner party at Chen\'s apartment in San Francisco. Karim did not attend the party and denied that it had occurred, but Chen commented that the idea that YouTube was founded after a dinner party "was probably very strengthened by marketing ideas around creating a story that was very digestible".'}]

In [17]:
question = "What is Albert Eistein famous for?"
context = get_context(question, top_k=1)
extract_answer(question, context)

[{'answer': 'his theories of special relativity and general relativity',
  'context': 'Albert Einstein is known for his theories of special relativity '
             'and general relativity. He also made important contributions to '
             'statistical mechanics, especially his mathematical treatment of '
             'Brownian motion, his resolution of the paradox of specific '
             'heats, and his connection of fluctuations and dissipation. '
             'Despite his reservations about its interpretation, Einstein also '
             'made contributions to quantum mechanics and, indirectly, quantum '
             'field theory, primarily through his theoretical studies of the '
             'photon.',
  'end': 86,
  'score': 0.9500371217727661,
  'start': 29}]


[{'score': 0.9500371217727661,
  'start': 29,
  'end': 86,
  'answer': 'his theories of special relativity and general relativity',
  'context': 'Albert Einstein is known for his theories of special relativity and general relativity. He also made important contributions to statistical mechanics, especially his mathematical treatment of Brownian motion, his resolution of the paradox of specific heats, and his connection of fluctuations and dissipation. Despite his reservations about its interpretation, Einstein also made contributions to quantum mechanics and, indirectly, quantum field theory, primarily through his theoretical studies of the photon.'}]

Let's run another question. This time for top 3 context passages from the retriever.

In [18]:
question = "Who was the first person to step foot on the moon?"
context = get_context(question, top_k=3)
extract_answer(question, context)

[{'answer': 'Armstrong',
  'context': 'The trip to the Moon took just over three days. After achieving '
             'orbit, Armstrong and Aldrin transferred into the Lunar Module, '
             'named Eagle, and after a landing gear inspection by Collins '
             'remaining in the Command/Service Module Columbia, began their '
             'descent. After overcoming several computer overload alarms '
             'caused by an antenna switch left in the wrong position, and a '
             'slight downrange error, Armstrong took over manual flight '
             'control at about 180 meters (590 ft), and guided the Lunar '
             'Module to a safe landing spot at 20:18:04 UTC, July 20, 1969 '
             '(3:17:04 pm CDT). The first humans on the Moon would wait '
             'another six hours before they ventured out of their craft. At '
             '02:56 UTC, July 21 (9:56 pm CDT July 20), Armstrong became the '
             'first human to set foot on the Moon.',

[{'score': 0.9998037815093994,
  'start': 71,
  'end': 80,
  'answer': 'Armstrong',
  'context': 'The trip to the Moon took just over three days. After achieving orbit, Armstrong and Aldrin transferred into the Lunar Module, named Eagle, and after a landing gear inspection by Collins remaining in the Command/Service Module Columbia, began their descent. After overcoming several computer overload alarms caused by an antenna switch left in the wrong position, and a slight downrange error, Armstrong took over manual flight control at about 180 meters (590 ft), and guided the Lunar Module to a safe landing spot at 20:18:04 UTC, July 20, 1969 (3:17:04 pm CDT). The first humans on the Moon would wait another six hours before they ventured out of their craft. At 02:56 UTC, July 21 (9:56 pm CDT July 20), Armstrong became the first human to set foot on the Moon.'},
 {'score': 0.695867121219635,
  'start': 240,
  'end': 246,
  'answer': 'Aldrin',
  'context': 'The first step was witnessed by at 

<br>

### 🚀 Challenge

Try adding a few more questions of your own in new cells below (reusing `get_context` and `extract_answer`). Experiment with different `top_k` values and observe:

- Does the retriever always return a context that actually contains the answer?
- How does the reader's confidence score change with harder or more ambiguous questions?
- What happens with questions that have no good answer in the dataset?

In [19]:
question = "What is the capital of France?"
context = get_context(question, top_k=1)
extract_answer(question, context)

[{'answer': 'Paris',
  'context': 'Most French rulers since the Middle Ages made a point of leaving '
             "their mark on a city that, contrary to many other of the world's "
             'capitals, has never been destroyed by catastrophe or war. In '
             'modernising its infrastructure through the centuries, Paris has '
             'preserved even its earliest history in its street map.[citation '
             'needed] At its origin, before the Middle Ages, the city was '
             'composed around several islands and sandbanks in a bend of the '
             'Seine; of those, two remain today: the île Saint-Louis, the île '
             'de la Cité; a third one is the 1827 artificially created île aux '
             'Cygnes. Modern Paris owes much to its late 19th century Second '
             "Empire remodelling by the Baron Haussmann: many of modern Paris' "
             'busiest streets, avenues and boulevards today are a result of '
             'that city re

[{'score': 3.590054087343475e-10,
  'start': 245,
  'end': 250,
  'answer': 'Paris',
  'context': 'Most French rulers since the Middle Ages made a point of leaving their mark on a city that, contrary to many other of the world\'s capitals, has never been destroyed by catastrophe or war. In modernising its infrastructure through the centuries, Paris has preserved even its earliest history in its street map.[citation needed] At its origin, before the Middle Ages, the city was composed around several islands and sandbanks in a bend of the Seine; of those, two remain today: the île Saint-Louis, the île de la Cité; a third one is the 1827 artificially created île aux Cygnes. Modern Paris owes much to its late 19th century Second Empire remodelling by the Baron Haussmann: many of modern Paris\' busiest streets, avenues and boulevards today are a result of that city renovation. Paris also owes its style to its aligned street-fronts, distinctive cream-grey "Paris stone" building ornamentation,

In [20]:
question = "Who developed the theory of relativity?"
context = get_context(question, top_k=3)
extract_answer(question, context)

[{'answer': 'Einstein',
  'context': 'The beginning of the 20th century brought the start of a '
             'revolution in physics. The long-held theories of Newton were '
             'shown not to be correct in all circumstances. Beginning in 1900, '
             'Max Planck, Albert Einstein, Niels Bohr and others developed '
             'quantum theories to explain various anomalous experimental '
             'results, by introducing discrete energy levels. Not only did '
             'quantum mechanics show that the laws of motion did not hold on '
             'small scales, but even more disturbingly, the theory of general '
             'relativity, proposed by Einstein in 1915, showed that the fixed '
             'background of spacetime, on which both Newtonian mechanics and '
             'special relativity depended, could not exist. In 1925, Werner '
             'Heisenberg and Erwin Schrödinger formulated quantum mechanics, '
             'which explained the precedi

[{'score': 0.9999581575393677,
  'start': 515,
  'end': 523,
  'answer': 'Einstein',
  'context': 'The beginning of the 20th century brought the start of a revolution in physics. The long-held theories of Newton were shown not to be correct in all circumstances. Beginning in 1900, Max Planck, Albert Einstein, Niels Bohr and others developed quantum theories to explain various anomalous experimental results, by introducing discrete energy levels. Not only did quantum mechanics show that the laws of motion did not hold on small scales, but even more disturbingly, the theory of general relativity, proposed by Einstein in 1915, showed that the fixed background of spacetime, on which both Newtonian mechanics and special relativity depended, could not exist. In 1925, Werner Heisenberg and Erwin Schrödinger formulated quantum mechanics, which explained the preceding quantum theories. The observation by Edwin Hubble in 1929 that the speed at which galaxies recede positively correlates with the

In [21]:
question = "What will the weather be in Frankfurt on 17 May 2040?"
context = get_context(question, top_k=3)
extract_answer(question, context)

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


[{'answer': 'mid 80s to low 90s',
  'context': 'The wet season begins some time in May, ending in mid-October. '
             'During this period, temperatures are in the mid 80s to low 90s '
             '(29–35 °C), accompanied by high humidity, though the heat is '
             'often relieved by afternoon thunderstorms or a sea breeze that '
             'develops off the Atlantic Ocean, which then allow lower '
             'temperatures, but conditions still remain very muggy. Much of '
             "the year's 55.9 inches (1,420 mm) of rainfall occurs during this "
             'period. Dewpoints in the warm months range from 71.9 °F (22.2 '
             '°C) in June to 73.7 °F (23.2 °C) in August.',
  'end': 125,
  'score': 5.454695042317326e-07,
  'start': 107},
 {'answer': 'southwest monsoon',
  'context': 'The onset of the southwest monsoon is anticipated to reach '
             'Punjab by May, but since the early 1970s the weather pattern has '
             'been irregular.

[{'score': 5.454695042317326e-07,
  'start': 107,
  'end': 125,
  'answer': 'mid 80s to low 90s',
  'context': "The wet season begins some time in May, ending in mid-October. During this period, temperatures are in the mid 80s to low 90s (29–35 °C), accompanied by high humidity, though the heat is often relieved by afternoon thunderstorms or a sea breeze that develops off the Atlantic Ocean, which then allow lower temperatures, but conditions still remain very muggy. Much of the year's 55.9 inches (1,420 mm) of rainfall occurs during this period. Dewpoints in the warm months range from 71.9 °F (22.2 °C) in June to 73.7 °F (23.2 °C) in August."},
 {'score': 8.656311886907908e-13,
  'start': 17,
  'end': 34,
  'answer': 'southwest monsoon',
  'context': 'The onset of the southwest monsoon is anticipated to reach Punjab by May, but since the early 1970s the weather pattern has been irregular. The spring monsoon has either skipped over the area or has caused it to rain so hard that floods 

In [22]:
question = "What will the weather be in Frankfurt on 17 May 2040?"
context = get_context(question, top_k=3)

for i, c in enumerate(context, start=1):
    print(f"\n--- CONTEXT {i} ---\n")
    print(c[:1000])


--- CONTEXT 1 ---

The wet season begins some time in May, ending in mid-October. During this period, temperatures are in the mid 80s to low 90s (29–35 °C), accompanied by high humidity, though the heat is often relieved by afternoon thunderstorms or a sea breeze that develops off the Atlantic Ocean, which then allow lower temperatures, but conditions still remain very muggy. Much of the year's 55.9 inches (1,420 mm) of rainfall occurs during this period. Dewpoints in the warm months range from 71.9 °F (22.2 °C) in June to 73.7 °F (23.2 °C) in August.

--- CONTEXT 2 ---

Detroit and the rest of southeastern Michigan have a humid continental climate (Köppen Dfa) which is influenced by the Great Lakes; the city and close-in suburbs are part of USDA Hardiness zone 6b, with farther-out northern and western suburbs generally falling in zone 6a. Winters are cold, with moderate snowfall and temperatures not rising above freezing on an average 44 days annually, while dropping to or below 0 °F

In [ ]:
# ...
# ...
# ...

<br>

---

### Clean Up

Now that we're done experimenting, let's delete the Pinecone index we created. This frees up the resources it was using (and, when using a paid plan, avoids being billed for an index we no longer need).

In [ ]:
# To delete the Pinecone index,
# uncomment the line below + run this code cell

# pc.delete_index(index_name)